# Clase 9 — Dashboards con Dash: fundamentos

## ¿Por qué te importa esto?

Hasta ahora has generado gráficos: los ejecutas, los miras, quizás los guardas como imagen. Eso funciona de maravilla para un informe cerrado. Pero en el momento en que alguien te dice "¿y si filtro por año?" o "¿puedo verlo yo mismo?", un gráfico estático se queda corto. Tendrías que volver a tu script, cambiar un valor, correrlo de nuevo y mandar otra imagen.

Un dashboard le da a esa persona el control. Abre una página en el navegador, mueve un control, y el gráfico se actualiza solo — sin que tú intervengas. **Dash** es la herramienta que te permite construir eso usando solo Python, sin tener que aprender HTML, CSS ni JavaScript a fondo.

## ⚠️ Nota técnica importante: Dash y los notebooks

Dash **no es una librería de gráficos que se muestra en una celda** — es un micro-framework que arranca un **servidor web** y sirve tu aplicación en una URL (normalmente `http://127.0.0.1:8050/`). Esto significa que, a diferencia de Matplotlib o Plotly, **no puedes simplemente ejecutar una celda y ver el resultado inline** sin más.

Tienes dos opciones para trabajar con Dash desde este material:

**Opción A — La forma "oficial" del curso (recomendada):** guarda el código en un archivo `.py` (por ejemplo `dash_basics.py`) y ejecútalo desde una terminal, ya sea en tu máquina local o en el IDE en la nube de Skills Network (Theia). Así es exactamente como lo hace el material original de IBM:

```
python dash_basics.py
```

La terminal te mostrará algo como `Dash is running on http://127.0.0.1:8050/`. Abres esa dirección en el navegador y ahí ves tu dashboard.

**Opción B — Ejecutarlo dentro de este mismo notebook (Colab o Jupyter local):** Dash normal bloquea la celda esperando conexiones si lo arrancas con `app.run()`. Las versiones recientes de Dash (2.11 en adelante, que es lo que instala `pip install dash` hoy) traen soporte nativo para notebooks: basta con llamar a `app.run(jupyter_mode="inline")` en vez de `app.run()`, sin instalar ningún paquete adicional ni cambiar `dash.Dash` por nada distinto.

Más abajo vas a encontrar el código del dashboard escrito de **ambas formas**: una celda de código con el contenido exacto que copiarías a un archivo `.py` (Opción A), y una celda opcional que lo ejecuta aquí mismo con `jupyter_mode="inline"` (Opción B). Usa la que te resulte más cómoda — el contenido del dashboard es idéntico en ambas.

## La analogía: el panel de control de un auto

Piensa en Dash como el tablero de un auto. Tú, como diseñador, decides qué instrumentos tiene: el velocímetro, la radio, las luces de aviso. El conductor no necesita saber cómo funciona el motor por dentro — solo interactúa con los controles que tú pusiste ahí.

Construir un dashboard con Dash es exactamente eso: tú decides qué "pantallas" (gráficos) y qué "botones" (controles) va a tener tu aplicación, y qué información muestra cada pantalla. En esta clase vas a construir el tablero más simple posible: solo pantallas, sin botones todavía. Los botones — la interactividad real con `@app.callback` — llegan en la próxima clase.

## Los tres bloques de toda app Dash

Toda aplicación Dash, por simple o compleja que sea, se construye con tres bloques que siempre aparecen en el mismo orden:

1. **Imports**: traes las librerías que vas a usar (Dash, sus componentes, y normalmente pandas y Plotly para preparar los datos y las figuras).
2. **`app.layout`**: defines la estructura visual de la página — qué aparece y en qué orden. Aquí es donde usas `html.Div`, `html.H1`, `html.P` y `dcc.Graph`.
3. **`app.run()`**: arrancas un servidor web local que sirve tu aplicación en el navegador.

Vamos a ver cada bloque con el ejemplo real de un dashboard de aerolíneas: un gráfico de pastel (pie chart) que muestra la proporción de vuelos por grupo de distancia.

### Bloque 1: los imports

```python
import pandas as pd
import plotly.express as px
import dash
from dash import dcc
from dash import html
```

Nada nuevo aquí si ya usaste pandas y Plotly Express: `pd` para manejar los datos, `px` para crear las figuras. Lo nuevo es `dash` y sus dos submódulos:

- `dcc` (**D**ash **C**ore **C**omponents): componentes "inteligentes" como gráficos, menús desplegables o campos de entrada.
- `html`: componentes que reflejan etiquetas HTML básicas — títulos, párrafos, divisiones — pero escritos como funciones de Python.

### Preparar los datos y la figura

```python
airline_data = pd.read_csv(
    'https://.../airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

data = airline_data.sample(n=500, random_state=42)

fig = px.pie(data, values='Flights', names='DistanceGroup',
             title='Distance group proportion by flights')
```

Nota que la figura (`fig`) se crea con Plotly Express, exactamente como en clases anteriores. **Dash no reemplaza a Plotly — lo usa.** Dash se encarga de "ponerlo en una página web e interactuar con el usuario"; Plotly sigue siendo el que dibuja el gráfico.

### Bloque 2: `app.layout`

```python
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Airline Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    html.P('Proportion of distance group (250 mile distance interval group) by flights.',
           style={'textAlign': 'center', 'color': '#F57241'}),
    dcc.Graph(figure=fig),
])
```

Línea por línea:

- `app = dash.Dash(__name__)` crea la aplicación. Es la línea que siempre va antes del layout.
- `app.layout = html.Div(children=[...])` define **todo** lo que se ve en la página. `html.Div` es un contenedor — como una caja que agrupa elementos — y recibe una lista (`children`) con los componentes que va a mostrar, en orden.
- `html.H1(...)` es el título principal. El parámetro `style` es un diccionario de CSS: aquí lo centras (`textAlign`), le das un color específico (`color`) y un tamaño de letra (`font-size`).
- `html.P(...)` es un párrafo de texto — la descripción del gráfico, también centrada y con su propio color.
- `dcc.Graph(figure=fig)` es el componente que efectivamente dibuja el gráfico. Le pasas la figura de Plotly que creaste antes con el parámetro `figure`.

Fíjate en el paralelismo: cada elemento visual de tu dashboard (título, descripción, gráfico) corresponde a un elemento dentro de esa lista `children`. Si quieres añadir algo más a la página, añades otro elemento a la lista.

### Bloque 3: `app.run()`

```python
if __name__ == '__main__':
    app.run()
```

Esto arranca el servidor. No genera una imagen ni abre una ventana por sí solo — deja un proceso corriendo que espera conexiones del navegador.

## Código completo — Opción A: versión para guardar como `.py` y ejecutar en terminal

Esta es la versión "canónica": copia el contenido de la siguiente celda a un archivo, por ejemplo `dash_basics.py`, y en una terminal (local o en el IDE de Skills Network) ejecuta:

```
python dash_basics.py
```

La terminal mostrará algo como `Dash is running on http://127.0.0.1:8050/`. Abre esa dirección en tu navegador para ver el título, la descripción y el gráfico de pastel. Mientras el proceso siga corriendo en la terminal, la página sigue disponible; para detenerlo usa `Ctrl+C`.

Ejecutar esta celda dentro del notebook **no producirá nada útil** (o incluso puede quedarse "colgada" esperando conexiones) porque `dash.Dash` está pensado para procesos independientes, no para celdas de Jupyter — por eso más abajo tienes la Opción B.

In [ ]:
# ============================================================
# VERSIÓN PARA GUARDAR COMO dash_basics.py Y EJECUTAR EN TERMINAL
# (python dash_basics.py)
# ============================================================

# Bloque 1: imports
import pandas as pd
import plotly.express as px
import dash
from dash import dcc
from dash import html

# Leer los datos de aerolíneas en un DataFrame de pandas
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

# Muestra aleatoria de 500 puntos. random_state fijo para reproducibilidad.
data = airline_data.sample(n=500, random_state=42)

# Creación del pie chart con Plotly Express
fig = px.pie(data, values='Flights', names='DistanceGroup',
             title='Distance group proportion by flights')

# Bloque 2: app.layout
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Airline Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    html.P('Proportion of distance group (250 mile distance interval group) by flights.',
           style={'textAlign': 'center', 'color': '#F57241'}),
    dcc.Graph(figure=fig),
])

# Bloque 3: app.run()
if __name__ == '__main__':
    app.run()


## Código completo — Opción B (opcional): ejecutar aquí mismo, sin salir del notebook

Si quieres ver el dashboard **sin salir del notebook** (por ejemplo en Google Colab), usa exactamente el mismo código de la Opción A, cambiando solo la última línea: `app.run(jupyter_mode="inline")` en vez de `app.run()`. No hace falta instalar nada adicional — esta capacidad ya viene incluida en `dash` desde la versión 2.11.

Esta celda es **opcional**: solo ejecútala si de verdad quieres ver el resultado renderizado dentro de esta notebook.

In [ ]:
# ============================================================
# VERSIÓN OPCIONAL PARA EJECUTAR DENTRO DEL NOTEBOOK (sin dependencias extra)
# ============================================================

import pandas as pd
import plotly.express as px
import dash
from dash import dcc
from dash import html

airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

data = airline_data.sample(n=500, random_state=42)

fig = px.pie(data, values='Flights', names='DistanceGroup',
             title='Distance group proportion by flights')

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Airline Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    html.P('Proportion of distance group (250 mile distance interval group) by flights.',
           style={'textAlign': 'center', 'color': '#F57241'}),
    dcc.Graph(figure=fig),
])

# jupyter_mode="inline" renderiza la app dentro de la salida de esta celda
app.run(jupyter_mode="inline")

## Puntos clave

- Toda app Dash tiene tres bloques: **imports**, **`app.layout`** y **`app.run()`**.
- `app.layout` recibe un `html.Div` con una lista `children` que define, en orden, todo lo que aparece en la página.
- `html.H1` y `html.P` son texto (título y párrafo); `dcc.Graph` es el componente que muestra una figura de Plotly mediante su parámetro `figure`.
- El parámetro `style` en los componentes HTML es un diccionario de CSS (`textAlign`, `color`, `font-size`, etc.) que controla su apariencia.
- Dash no reemplaza a Plotly: las figuras se siguen creando con `plotly.express` o `plotly.graph_objects`; Dash solo las coloca dentro de una página web interactiva.
- En un entorno normal, corres la app con `python archivo.py` y la abres en el navegador usando el puerto que indica la terminal (normalmente `8050`). Dentro de un notebook, usa `app.run(jupyter_mode="inline")` en vez de `app.run()` — Dash 2.11+ ya lo trae incluido, sin instalar nada adicional.
- Los botones y controles interactivos (`@app.callback`) llegan en la próxima clase — hoy solo construimos "pantallas".